# ⚙️ Módulos y Librerías: Eliminación de Archivos (`os.remove`)

| Propiedad | Detalle |
| :--- | :--- |
| 📍 **Módulo** | Módulos, Paquetes y Librerías / Módulo OS |
| 🎯 **Objetivo** | Aprender a eliminar archivos individuales del sistema de archivos mediante `os.remove()`, distinguiendo su uso frente al borrado de directorios y gestionando excepciones clave como `FileNotFoundError` e `IsADirectoryError`. |
| 🛠️ **Conceptos** | `os.remove()`, `Eliminación de Ficheros`, `FileNotFoundError`, `IsADirectoryError`, `os.path.isfile()` |

---

### 💡 El Método `os.remove()`

La función **`os.remove(path)`** (o su alias `os.unlink(path)`) elimina permanentemente un **archivo** en la ruta especificada. Es importante destacar que **no envía los archivos a la papelera de reciclaje** y no puede utilizarse para eliminar directorios.

---

### 🗂️ Consideraciones y Excepciones Comunes

```text
[ Fichero: 'datos.txt' ] ──( os.remove )──> ✅ Eliminado Permanentemente
[ Carpeta: 'mi_carpeta' ] ──( os.remove )──> ❌ IsADirectoryError (Usar os.rmdir)
```

| Situación | Comportamiento / Error | Solución Recomendada |
| :--- | :--- | :--- |
| 📄 El archivo no existe | Lanza `FileNotFoundError`. | Comprobar existencia con `os.path.exists()` o `os.path.isfile()`. |
| 📁 La ruta es un directorio | Lanza `IsADirectoryError` (o `PermissionError` en Windows). | Utilizar `os.rmdir()` o `shutil.rmtree()` para directorios. |
| 🔒 Sin permisos / En uso | Lanza `PermissionError`. | Verificar que el archivo no esté abierto por otro proceso. |
```

In [ ]:
import os
from typing import Any

# 🛰️ [1] Banner visual de inicio de ejecución
print("=" * 80)
print("🐍 MÓDULO OS | Eliminación de Archivos con os.remove()")
print("=" * 80)

# ================================================================================
# 🧪 1. ELIMINACIÓN BÁSICA DE UN ARCHIVO CON VERIFICACIÓN PREVIA
# ================================================================================
# Creación de un archivo de prueba y borrado seguro con os.remove().
# ================================================================================

print("\n--- 🚀 [PASO 1]: Creación y Eliminación Controlada de un Fichero ---")

target_file: str = "archivo_a_eliminar.txt"

# ⚙️ [1] Crear un archivo temporal para la demostración
with open(target_file, "w", encoding="utf-8") as temp_file:
    temp_file.write("Contenido temporal destinado a ser eliminado.")

print(f"📋 Estado previo: ¿Existe '{target_file}'?: {os.path.exists(target_file)}")

try:
    # ⚙️ [2] Validar que el elemento existe y es un archivo antes de borrar
    if os.path.isfile(target_file):
        # 🔌 [1] Eliminar el archivo especificado
        os.remove(target_file)
        print(f"  ✅ Archivo '{target_file}' eliminado exitosamente con os.remove().")
    else:
        print(f"  ⚠️ El elemento '{target_file}' no existe o no es un archivo.")

finally:
    # 🔌 [2] Confirmar la eliminación del archivo
    print(f"📋 Estado posterior: ¿Existe '{target_file}'?: {os.path.exists(target_file)}")

## 🧱 Control de Excepciones Comunes en `os.remove()`

### 💡 Captura de FileNotFoundError e IsADirectoryError:
Tratar de borrar un archivo inexistente o intentar aplicar `os.remove()` sobre una carpeta provocará un fallo en la ejecución si no se gestionan las excepciones correctamente.

In [ ]:
# ================================================================================
# 🧪 2. MANEJO DE ERRORES AL ELIMINAR ELEMENTOS
# ================================================================================
print("\n--- 🚀 [PASO 2]: Captura de Errores (FileNotFoundError e IsADirectoryError) ---")

non_existent_file: str = "fichero_fantasma.log"
demo_folder: str = "carpeta_prueba_remove"

# Crear una carpeta para forzar la excepción al intentar borrarla con os.remove()
if not os.path.exists(demo_folder):
    os.mkdir(demo_folder)

try:
    # ⚙️ [1] Intento de eliminar un archivo que no existe
    print(f"📋 Intento 1: Borrando archivo inexistente '{non_existent_file}'...")
    os.remove(non_existent_file)

except FileNotFoundError as error:
    print("  ❌ Excepción capturada: El archivo a eliminar no fue encontrado.")
    print(f"  🔍 Detalle técnico: {error}")

try:
    # ⚙️ [2] Intento de eliminar un directorio con os.remove()
    print(f"\n📋 Intento 2: Usando os.remove() sobre la carpeta '{demo_folder}'...")
    os.remove(demo_folder)

except (IsADirectoryError, PermissionError) as error:
    print("  ❌ Excepción capturada: os.remove() no se puede aplicar sobre directorios.")
    print(f"  🔍 Detalle técnico: {error}")

finally:
    if os.path.exists(demo_folder):
        os.rmdir(demo_folder)
        print("  🔒 Carpeta de prueba limpiada con os.rmdir().")

## 🛒 Ejercicios Prácticos Avanzados Integrados

---

### 🧪 Ejercicio 1: Limpiador de Archivos Temporales y Logs Antiguos

**Escenario Empresarial:** Se necesita una función de mantenimiento que reciba una lista de rutas de archivos temporales (`.tmp`, `.bak`, `.log`), verifique si existen y sean ficheros válidos, y proceda a su eliminación segura registrando el informe final de ejecución.

In [ ]:
# ================================================================================
# 🧪 3. EJERCICIO PRÁCTICO 1: PURGADOR SEGURO DE ARCHIVOS TEMPORALES
# ================================================================================
print("\n--- 🧪 [PASO 3.1]: Ejercicio 1 - Limpiador Idempotente de Ficheros ---")


def purge_temporary_files(file_paths: list[str]) -> dict[str, Any]:
    """Elimina una lista de archivos de forma segura y genera un reporte de resultados.

    Args:
        file_paths (list[str]): Lista de rutas de archivos a eliminar.

    Returns:
        dict[str, Any]: Informe con archivos borrados, no encontrados y fallidos.
    """
    report: dict[str, Any] = {
        "deleted": [],
        "not_found": [],
        "failed": []
    }

    for path in file_paths:
        try:
            # [1] Verificar si la ruta corresponde a un archivo válido
            if not os.path.exists(path):
                report["not_found"].append(path)
                print(f"  ⚠️ Omitido: '{path}' (No existe)")
                continue

            if os.path.isdir(path):
                report["failed"].append({"path": path, "reason": "Es un directorio, no un archivo"})
                print(f"  ❌ Falló: '{path}' (Es una carpeta)")
                continue

            # [2] Eliminar el archivo
            os.remove(path)
            report["deleted"].append(path)
            print(f"  ✅ Eliminado: '{path}'")

        except PermissionError:
            report["failed"].append({"path": path, "reason": "Permisos insuficientes o archivo en uso"})
            print(f"  ❌ Falló: '{path}' (Sin permisos / En uso)")

        except OSError as error:
            report["failed"].append({"path": path, "reason": str(error)})
            print(f"  ❌ Falló: '{path}' ({error})")

    return report


# Crear archivos de prueba para la ejecución
test_files: list[str] = ["temp_cache.tmp", "old_session.bak"]
for file_name in test_files:
    with open(file_name, "w", encoding="utf-8") as f:
        f.write("Cache data")

# Crear una carpeta para probar la falla intencional
invalid_folder: str = "invalid_file_target"
os.makedirs(invalid_folder, exist_ok=True)

# Lista completa a procesar
candidates: list[str] = test_files + ["missing_file.log", invalid_folder]

print("📋 Ejecutando purga de archivos temporales...")
purge_summary: dict[str, Any] = purge_temporary_files(candidates)

print("\n📋 Resumen del Proceso de Purga:")
print(f"  🔹 Archivos eliminados ({len(purge_summary['deleted'])}): {purge_summary['deleted']}")
print(f"  🔹 Archivos no encontrados ({len(purge_summary['not_found'])}): {purge_summary['not_found']}")
print(f"  🔹 Errores de eliminación ({len(purge_summary['failed'])}): {purge_summary['failed']}")

# Limpieza final de elementos de prueba residuales
if os.path.exists(invalid_folder):
    os.rmdir(invalid_folder)

# --------------------------------------------------------------------------------
# 🏁 [PASO 4]: Cierre de Ejecución
# --------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("✅ [FIN DE EJECUCIÓN] Lección de os.remove() finalizada.")
print("=" * 80)